In [1]:
%cd ../src

/app/src


In [2]:
%%writefile feature_engineering/feature_pipeline.py 
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any
import json

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from src.feature_engineering.holiday_feature import add_holiday_features
from src.feature_engineering.date_features import add_date_features
from src.preprocessing.parking_resampler import ParkingResampler
from src.preprocessing.parking_sks_merger import ParkingSKSMerger
from src.preprocessing.parking_calendar_merger import ParkingCalendarMerger
from src.preprocessing.parking_imputer import ParkingImputer
from src.feature_engineering.parking_capacity_corrector import ParkingSpacesCorrector
from src.feature_engineering.day_features import DayFeaturesCreator
from src.feature_engineering.rolling_features_creator import RollingFeaturesCreator
from src.feature_engineering.expanding_features_creator import ExpandingFeaturesCreator
from src.feature_engineering.lag_features_creator import LagFeaturesCreator
from src.feature_engineering.cannibalisation_adder import CannibalisationFeaturesCreator

@dataclass
class FeaturePipelineConfig:
    """All knobs for the feature pipeline. Round-trips through JSON."""

    #merge / resample rules
    agg_rules: dict = field(default_factory=lambda: {
        'spaces_left': 'mean',
        # 'trend': 'mean',  # weird, semi-random — leave out for now
    })

    # parking lot expansions 
    expansions: dict = field(default_factory=lambda: {
        "Parking Wrońskiego": '2025-05-21',
    })
    size_before_expansion: dict = field(default_factory=lambda: {
        "Parking Wrońskiego": 192,
    })

    # feature dicts: {id: (suffix, type, window, [cols])}
    # 1 hour = 12 records @ 5min freq; 1 day = 288; 7 days = 288*7
    rolling_features: dict = field(default_factory=lambda: {
        1: ("delta_1h",        "delta", 12,      ["spaces_left"]),
        2: ("Rolling_mean_1d", "mean",  288,     ["spaces_left"]),
        3: ("Rolling_mean_7d", "mean",  288 * 7, ["spaces_left"]),
        4: ("Rolling_std_1d",  "std",   288,     ["spaces_left"]),
        5: ("Rolling_std_7d",  "std",   288 * 7, ["spaces_left"]),
        6: ("Rolling_max_7d",  "max",   288 * 7, ["spaces_left"]),
        7: ("Rolling_min_7d",  "min",   288 * 7, ["spaces_left"]),
    })

    expanding_features: dict = field(default_factory=lambda: {
        1: ("cumulative_mean",  "mean", ["spaces_left"]),
        2: ("cumulative_max",   "max",  ["spaces_left"]),
        3: ("cumulative_min",   "min",  ["spaces_left"]),
        4: ("cumulative_trend", "std",  ["spaces_left"]),
    })

    lag_features: dict = field(default_factory=lambda: {
        1: ("lag_1h", 12, ["spaces_left", "utilization_rate"]),
        2: ("lag_7d", 288 * 7, [
            "spaces_left", "utilization_rate",
            "spaces_left_Rolling_mean_7d", "spaces_left_Rolling_std_7d",
            "spaces_left_Rolling_max_7d",  "spaces_left_Rolling_min_7d",
        ]),
    })

    cannibalisation_features: dict = field(default_factory=lambda: {
        1: ("cann_closest_parking",        "closest",        ["spaces_left", "spaces_left_lag_7d", "utilization_rate_lag_1h", "utilization_rate_lag_7d", "spaces_left_delta_1h"]),
        2: ("cann_second_closest_parking", "second_closest", ["spaces_left", "utilization_rate_lag_1h"]),
        3: ("cann_third_closest_parking",  "third_closest",  ["spaces_left", "utilization_rate_lag_1h"]),
        4: ("cann_fourth_closest_parking", "fourth_closest", ["spaces_left", "utilization_rate_lag_1h"]),
    })

    frequency_minutes: int = 5
    convert_to_32_bit: bool = True
    shift_guard_periods: int = 15
    cannibalisation_guard_periods: int = 15
    copy: bool = False
    ffill_limit: int = 60
    cols_to_impute: list = field(default_factory=lambda:
        ['spaces_left', 'active_users', 'sks_dist_to_users_ratio']
    )

    # --- (de)serialization -------------------------------------------
    # Feature dicts that need int keys + tuple values restored after JSON load.
    _INDEXED_FEATURE_DICTS = (
        'rolling_features',
        'expanding_features',
        'lag_features',
        'cannibalisation_features',
    )

    @classmethod
    def from_json(cls, path: str | Path) -> "FeaturePipelineConfig":
        with open(path, 'r', encoding='utf-8') as f:
            data: dict[str, Any] = json.load(f)

        # JSON has no int keys and no tuples — rebuild them.
        for key in cls._INDEXED_FEATURE_DICTS:
            if key in data and data[key] is not None:
                data[key] = {int(k): tuple(v) for k, v in data[key].items()}

        return cls(**data)

    def to_json(self, path: str | Path) -> None:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(asdict(self), f, indent=2, ensure_ascii=False)



# Builder

def build_feature_pipeline(
    config: FeaturePipelineConfig,
    df_sks_users,
    df_parkings,
    df_calendar,
) -> Pipeline:
    # Prep transformers
    parking_resampler = ParkingResampler(
        convert_to_32=config.convert_to_32_bit,
        agg_rules=config.agg_rules,
    )

    # use kw_args to pass additional arguments to the function transformer
    holiday_transformer = FunctionTransformer(
        add_holiday_features, kw_args={'copy': config.copy}
    )
    date_transformer = FunctionTransformer(
        add_date_features, kw_args={'copy': config.copy}
    )

    parking_sks_merger = ParkingSKSMerger(
        sks_df=df_sks_users,
        parkings_df=df_parkings,
        freq_minutes=config.frequency_minutes,
        convert_to_32=config.convert_to_32_bit,
        copy=config.copy,
    )
    parking_calendar_merger = ParkingCalendarMerger(
        calendar_df=df_calendar,
        freq_minutes=config.frequency_minutes,
        convert_to_32=config.convert_to_32_bit,
        copy=config.copy,
    )
    parking_imputer = ParkingImputer(
        cols_to_impute=config.cols_to_impute,
        freq_minutes=config.frequency_minutes,
        ffill_limit=config.ffill_limit,
        convert_to_32=config.convert_to_32_bit,
        copy=config.copy,
    )
    parkings_capacity_corrector = ParkingSpacesCorrector(
        df_parkings,
        config.expansions,
        config.size_before_expansion,
        convert_to_32=config.convert_to_32_bit,
        copy=config.copy,
    )
    day_feature_creator = DayFeaturesCreator(
        df_parkings,
        convert_to_32=config.convert_to_32_bit,
        copy=config.copy,
    )
    rolling_features_creator = RollingFeaturesCreator(
        rolling_features=config.rolling_features,
        convert_to_32=config.convert_to_32_bit,
        shift_guard_periods=config.shift_guard_periods,
        copy=config.copy,
    )
    lag_features_creator = LagFeaturesCreator(
        config.lag_features,
        convert_to_32=config.convert_to_32_bit,
        copy=config.copy,
    )
    cannibalisation_features_creator = CannibalisationFeaturesCreator(
        df_parkings,
        config.cannibalisation_features,
        cannibalisation_guard=config.cannibalisation_guard_periods,
        convert_to_32=config.convert_to_32_bit,
        copy=config.copy,
    )

    # Transformer pipeline
    feature_pipeline = Pipeline([
        ('resampler',                parking_resampler),
        ('date_features',            date_transformer),
        ('holiday_features',         holiday_transformer),
        ('sks_merger',               parking_sks_merger),
        ('calendar_merger',          parking_calendar_merger),
        ('imputer',                  parking_imputer),
        ('capacity_corrector',       parkings_capacity_corrector),
        ('day_features',             day_feature_creator),
        ('rolling_features',         rolling_features_creator),
        ('lag_features',             lag_features_creator),
        ('cannibalisation_features', cannibalisation_features_creator),
    ])
    return feature_pipeline


Overwriting feature_engineering/feature_pipeline.py


In [4]:
from src.feature_engineering.feature_pipeline import build_feature_pipeline , FeaturePipelineConfig

In [5]:
import pandas as pd
from config import DATA_DIR
df_parkings = pd.read_parquet(DATA_DIR / 'parkings.parquet')
df_parkings_availabilities = pd.read_parquet(DATA_DIR / 'parking_availabilities.parquet')
df_sks_users = pd.read_parquet(DATA_DIR / 'sks_users.parquet')
df_calendar = pd.read_parquet(DATA_DIR / 'calendar.parquet')

In [6]:
df_min = df_parkings_availabilities.groupby('parking_id')['spaces_left'].max()
df_min.name = 'max_spaces_left'
df_parkings = df_parkings.merge(df_min, left_on='id', right_index=True)

print(df_parkings[['name', 'places', 'max_spaces_left']])

                 name  places  max_spaces_left
0        Architektura      75               84
1  Parking Wrońskiego     207              207
2             Polinka      54               61
3           D20 - D21      76               49
4  GEO LO1 Geocentrum     301              267


In [7]:
import pandas as pd

# Manualy collected distances based on google maps
manual_data = [
    {'id': 7, 'd_to_7': 0,    'd_to_4': 2600, 'd_to_2': 2100, 'd_to_5': 2100, 'd_to_6': 2500, 'dist_to_sks': 2500},
    {'id': 4, 'd_to_7': 2600, 'd_to_4': 0,    'd_to_2': 800,  'd_to_5': 350,  'd_to_6': 1800, 'dist_to_sks': 42},
    {'id': 2, 'd_to_7': 2100, 'd_to_4': 500,  'd_to_2': 0,    'd_to_5': 650,  'd_to_6': 1800, 'dist_to_sks': 450},
    {'id': 5, 'd_to_7': 2100, 'd_to_4': 350,  'd_to_2': 650,  'd_to_5': 0,    'd_to_6': 2400, 'dist_to_sks': 400},
    {'id': 6, 'd_to_7': 2500, 'd_to_4': 1800, 'd_to_2': 1800, 'd_to_5': 2400, 'd_to_6': 0,    'dist_to_sks': 1500}
]

processed_rows = []

for entry in manual_data:
    current_id = entry['id']
    
    # Create a simple list of neighbors: (distance, neighbor_id)
    neighbors = []
    for key, dist in entry.items():
        if key.startswith('d_to_') and key != f'd_to_{current_id}':
            neighbor_id = int(key.split('_')[-1]) # Extract 4 from 'd_to_4'
            neighbors.append((dist, neighbor_id))
            
    # Sort closest to farthest
    neighbors.sort()
    
    # Build the simple row
    processed_rows.append({
        'id': current_id,
        'closest_id': neighbors[0][1],
        'second_closest_id': neighbors[1][1],
        'third_closest_id': neighbors[2][1],
        'fourth_closest_id': neighbors[3][1],
        'distance_to_sks': entry['dist_to_sks']
    })

df_features = pd.DataFrame(processed_rows)
df_parkings = df_parkings.merge(df_features, on='id', how='left')

# Check result
df_parkings.head()

,id,symbol,type,name,open_hour,close_hour,places,geo_lan,geo_lat,is_active,is_visible,address,created_at,updated_at,max_spaces_left,closest_id,second_closest_id,third_closest_id,fourth_closest_id,distance_to_sks
0,7,E01,O,Architektura,06:00:00,22:30:00,75,17.054167,51.118736,True,True,"Bolesława Prusa 53/55, 50-317 Wrocław\r\n",2025-02-03 07:51:16.241000+00:00,2025-12-06 19:46:06.731000+00:00,84,2,5,6,4,2500
1,4,WRO,O,Parking Wrońskiego,06:00:00,22:00:00,207,17.055565,51.108963,True,True,"Hoene-Wrońskiego 10, 50-376 Wrocław",2025-02-03 07:51:16.185000+00:00,2025-12-06 19:46:06.218000+00:00,207,5,2,6,7,42
2,2,C13,O,Polinka,None,None,54,17.058468,51.107390,True,True,"wybrzeże Stanisława Wyspiańskiego 25, 50-370 W...",2025-02-03 07:51:16.210000+00:00,2025-12-06 19:46:06.343000+00:00,61,4,5,6,7,450
3,5,D20,O,D20 - D21,06:00:00,22:30:00,76,17.059677,51.110050,True,True,"Janiszewskiego 8, 50-372 Wrocław\r\n",2025-02-03 07:51:16.222000+00:00,2025-12-06 19:46:06.478000+00:00,49,4,2,7,6,400
4,6,GEO-L,O,GEO LO1 Geocentrum,06:00:00,22:30:00,301,17.055334,51.104164,True,True,"Na Grobli 15, 50-421 Wrocław\r\n",2025-02-03 07:51:16.232000+00:00,2025-12-06 19:46:06.602000+00:00,267,2,4,5,7,1500


In [10]:
config = FeaturePipelineConfig()
pipeline = build_feature_pipeline(config, df_sks_users, df_parkings, df_calendar)
config.to_json("config/default.json")

In [11]:
pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',1000)

In [12]:
df_processed = pipeline.fit_transform(df_parkings_availabilities)
df_processed.head(10)

,parking_id,measured_at,spaces_left,year,month,quarter,day_in_the_month,week_of_the_year,day_of_week,is_weekend,hour,minute,is_working_hour,is_rush_hour,is_holiday,days_since_holiday,days_until_holiday,is_long_weekend_bridge,distance_to_sks,active_users,sks_dist_to_users_ratio,day_type_id,spaces_left_is_imputed,active_users_is_imputed,sks_dist_to_users_ratio_is_imputed,overbooked,overbooked_spaces,is_expanded,total_capacity,is_open,spaces_left_delta_1h,spaces_left_Rolling_mean_1d,spaces_left_Rolling_mean_7d,spaces_left_Rolling_std_1d,spaces_left_Rolling_std_7d,spaces_left_Rolling_max_7d,spaces_left_Rolling_min_7d,spaces_left_lag_1h,utilization_rate_lag_1h,spaces_left_lag_7d,utilization_rate_lag_7d,spaces_left_Rolling_mean_7d_lag_7d,spaces_left_Rolling_std_7d_lag_7d,spaces_left_Rolling_max_7d_lag_7d,spaces_left_Rolling_min_7d_lag_7d,spaces_left_cann_closest_parking,spaces_left_lag_7d_cann_closest_parking,utilization_rate_lag_1h_cann_closest_parking,utilization_rate_lag_7d_cann_closest_parking,spaces_left_delta_1h_cann_closest_parking,spaces_left_cann_second_closest_parking,utilization_rate_lag_1h_cann_second_closest_parking,spaces_left_cann_third_closest_parking,utilization_rate_lag_1h_cann_third_closest_parking,spaces_left_cann_fourth_closest_parking,utilization_rate_lag_1h_cann_fourth_closest_parking
0,2,2025-02-03 07:50:00+00:00,4,2025,2,1,3,6,0,0,7,50,1,1,0,28.0,76.0,0,450.0,4,0.89,2,0,0,0,0,0,0,61,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,2025-02-03 07:50:00+00:00,0,2025,2,1,3,6,0,0,7,50,1,1,0,28.0,76.0,0,42.0,4,9.52,2,0,0,0,0,0,0,192,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7,2025-02-03 07:50:00+00:00,57,2025,2,1,3,6,0,0,7,50,1,1,0,28.0,76.0,0,2500.0,4,0.16,2,0,0,0,0,0,0,84,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6,2025-02-03 07:50:00+00:00,166,2025,2,1,3,6,0,0,7,50,1,1,0,28.0,76.0,0,1500.0,4,0.27,2,0,0,0,0,0,0,267,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2025-02-03 07:50:00+00:00,0,2025,2,1,3,6,0,0,7,50,1,1,0,28.0,76.0,0,400.0,4,1.00,2,0,0,0,0,0,0,49,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,7,2025-02-03 07:55:00+00:00,57,2025,2,1,3,6,0,0,7,55,1,1,0,28.0,76.0,0,2500.0,3,0.12,2,0,0,0,0,0,0,84,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,4,2025-02-03 07:55:00+00:00,0,2025,2,1,3,6,0,0,7,55,1,1,0,28.0,76.0,0,42.0,3,7.14,2,0,0,0,0,0,0,192,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,6,2025-02-03 07:55:00+00:00,160,2025,2,1,3,6,0,0,7,55,1,1,0,28.0,76.0,0,1500.0,3,0.20,2,0,0,0,0,0,0,267,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2,2025-02-03 07:55:00+00:00,4,2025,2,1,3,6,0,0,7,55,1,1,0,28.0,76.0,0,450.0,3,0.67,2,0,0,0,0,0,0,61,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,5,2025-02-03 07:55:00+00:00,0,2025,2,1,3,6,0,0,7,55,1,1,0,28.0,76.0,0,400.0,3,0.75,2,0,0,0,0,0,0,49,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
